In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from src.data_loader import load_insurance_data
from src.modeling import train_and_evaluate_all_models

# Load clean version 2 data matrix
df = load_insurance_data('../data/MachineLearningRating_v3.txt')

# FEATURE ENGINEERING
# 1. Isolate the target sub-population for Severity Prediction (Claims > 0)
severity_df = df[df['TotalClaims'] > 0].copy()

# 2. Build explicit predictive features (e.g., Vehicle Age)
if 'VehicleIntroYear' in severity_df.columns:
    severity_df['VehicleAge'] = 2026 - severity_df['VehicleIntroYear']
else:
    severity_df['VehicleAge'] = 5 # Standard fallback imputation

# 3. Vectorize critical features via quick Label Encoding
categorical_cols = ['Province', 'VehicleType', 'Gender', 'make']
for col in categorical_cols:
    severity_df[col] = severity_df[col].astype('category').cat.codes

# Define final Feature Matrix and Target Variable Array
features = ['VehicleAge', 'CustomValueEstimate', 'Province', 'VehicleType', 'Gender', 'make']
X = severity_df[features].fillna(severity_df.median(numeric_only=True))
y = severity_df['TotalClaims']

# Split Data into 80:20 Train/Test partitions
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)